In [1]:
import gradio as gr
import unicodedata, torch, faiss
import numpy as np, json
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

torch.set_num_threads(2)

# Modèles
print("Chargement des modèles...")
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cpu")
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)
qa_model.eval()

# KB + Index
with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)
index = faiss.read_index("ensa_faiss_v3.index")

print(f"Prêt — {len(ensa_data)} entrées, {index.ntotal} vecteurs")

Chargement des modèles...


C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Prêt — 320 entrées, 320 vecteurs


In [34]:
import random

SEUIL_COSINUS = 0.40

def normaliser(texte):
    return unicodedata.normalize('NFD', texte)\
                      .encode('ascii', 'ignore')\
                      .decode('utf-8')

def retrieve(query, top_k=3):
    qvec = embedding_model.encode([normaliser(query)]).astype(np.float32)
    faiss.normalize_L2(qvec)
    scores, indices = index.search(qvec, top_k)
    return [{
        "score_cosinus": float(scores[0][i]),
        "answer": ensa_data[idx]["answer"],
        "question": ensa_data[idx]["question"],
        "category": ensa_data[idx]["category"]
    } for i, idx in enumerate(indices[0]) if idx < len(ensa_data)]

def read_answer(question, context):
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )
    score = float(torch.softmax(outputs.start_logits, dim=1).max())
    return answer, score

def calculer_confiance(score_cosinus, score_qa):
    if score_cosinus < SEUIL_COSINUS:
        return 0
    return min(round((0.6 * score_cosinus + 0.4 * score_qa) * 100), 99)

    import random

REPONSES_SOCIALES = {
    "salutation": [
        "Hello! I'm the ENSA Fès intelligent assistant. Feel free to ask me anything about programs, admissions, clubs, or contacts!",
        "Hi there! Welcome to the ENSA Fès chatbot. What would you like to know about our school?",
        "Hey! Great to see you here. I'm ready to answer all your questions about ENSA Fès.",
        "Bonjour! Je suis l'assistant intelligent de l'ENSA Fès. Comment puis-je vous aider aujourd'hui?",
        "Salut! Bienvenue sur le chatbot de l'ENSA Fès. Posez-moi vos questions sur l'école!",
        "Salam! Ana mosa3id ENSA Fès. Kifach nqder n3awnek?",
    ],
    "comment_ca_va": [
        "I'm doing great, thank you for asking! Ready to help you with anything about ENSA Fès.",
        "All systems running smoothly! What can I help you with today?",
        "Très bien merci! Et vous? En tout cas, je suis là pour répondre à toutes vos questions sur l'ENSA Fès.",
        "Ca va bien! Je suis opérationnel et prêt à vous aider. Quelle est votre question?",
        "Running at full capacity! Ask me anything about ENSA Fès.",
    ],
    "merci": [
        "You're very welcome! Don't hesitate to ask more questions about ENSA Fès.",
        "Happy to help! Is there anything else you'd like to know?",
        "My pleasure! Feel free to explore more topics about ENSA Fès.",
        "De rien! C'est avec plaisir que je vous aide. Avez-vous d'autres questions?",
        "Avec plaisir! N'hésitez pas si vous avez d'autres questions sur l'ENSA Fès.",
        "Bghit nsa3dek! Wash kayn chi haja okhra?",
    ],
    "au_revoir": [
        "Goodbye! Good luck with your studies and future at ENSA Fès!",
        "See you soon! Don't hesitate to come back if you have more questions.",
        "Au revoir! Bonne continuation et bonne chance dans vos études!",
        "Bonne journée! N'hésitez pas à revenir si vous avez d'autres questions.",
        "Take care! ENSA Fès is always here for you.",
        "Bye bye! Bslama w bonne chance!",
    ],
    "identite": [
        "I'm an intelligent chatbot built specifically for ENSA Fès. I use a RAG pipeline combining a semantic BERT retriever and a RoBERTa reader to give you accurate answers about the school.",
        "I'm the ENSA Fès AI Assistant! I can answer questions about programs, admissions, clubs, internships, contacts, and much more. Just ask!",
        "Je suis l'assistant intelligent de l'ENSA Fès, basé sur une architecture RAG — je combine un moteur de recherche sémantique et un modèle de lecture pour vous donner des réponses précises.",
        "Think of me as your personal ENSA Fès guide! I know everything about the school — programs, professors, clubs, admissions, and more.",
    ],
    "blague": [
        "Why do programmers prefer dark mode? Because light attracts bugs! But seriously, how can I help you with ENSA Fès?",
        "I'd tell you a joke about engineers, but I'm still debugging it. Anyway, what can I help you with?",
        "Ha! Good one. Now, shall we talk about ENSA Fès?",
    ],
    "felicitations": [
        "Thank you so much! I'm just doing my job. Now, what can I help you with?",
        "That's very kind of you! I'm here to make your ENSA Fès experience easier.",
        "Merci beaucoup! C'est un plaisir de vous aider. Avez-vous d'autres questions?",
    ],
    "aide": [
        "Of course! I can help you with: admissions, engineering programs, internships, student clubs, campus contacts, tuition fees, academic calendar, and much more. Just ask!",
        "Sure! Here's what I can do for you:\n- Answer questions about ENSA Fès programs\n- Explain the admission process\n- Tell you about student clubs and activities\n- Provide contact information\n- And much more! What do you need?",
        "Bien sûr! Je peux vous aider avec: les filières, les admissions, les stages, les clubs, les contacts, les frais de scolarité, et bien plus encore!",
    ],
    "Mok":[
        "Jme3 kerek 3eye9ti a si zebi ","Ana mamak ",
    ],
}

def detecter_social(query):
    q = query.lower().strip().rstrip("!?.,")

    # Salutations
    if any(s in q for s in [
        "hello","hi","hey","bonjour","salut","salam","bonsoir",
        "good morning","good evening","good afternoon","ahlan","marhaba"
    ]):
        return random.choice(REPONSES_SOCIALES["salutation"])

    # Comment ça va
    if any(s in q for s in [
        "how are you","comment ca va","comment vas","ca va",
        "what's up","how is it going","how do you do",
        "labas","kif nta","kifash"
    ]):
        return random.choice(REPONSES_SOCIALES["comment_ca_va"])

    # Merci
    if any(s in q for s in [
        "thank","merci","شكرا","choukran","appreciate",
        "well done","great job","good job","amazing"
    ]):
        return random.choice(REPONSES_SOCIALES["merci"])

    # Félicitations
    if any(s in q for s in [
        "bravo","excellent","perfect","parfait","super",
        "awesome","incredible","impressive","mazing"
    ]):
        return random.choice(REPONSES_SOCIALES["felicitations"])

    # Au revoir
    if any(s in q for s in [
        "bye","goodbye","au revoir","bonne journee","bonne nuit",
        "see you","à bientôt","bslama","ciao","take care"
    ]):
        return random.choice(REPONSES_SOCIALES["au_revoir"])

    # Identité
    if any(s in q for s in [
        "who are you","what are you","tu es qui","vous etes",
        "what can you do","que peux","tell me about yourself",
        "introduce yourself","présente toi","c'est quoi toi"
    ]):
        return random.choice(REPONSES_SOCIALES["identite"])

    # Blague
    if any(s in q for s in [
        "joke","blague","funny","tell me something","make me laugh",
        "fais moi rire"
    ]):
        return random.choice(REPONSES_SOCIALES["blague"])

    # Aide générale
    if any(s in q for s in [
        "help","aide","how can you help","que peux-tu faire",
        "what do you know","what can you tell","assist"
    ]):
        return random.choice(REPONSES_SOCIALES["aide"])

    # Aide générale
    if any(s in q for s in [
        "afeen a w9","wach nta zamel"
    ]):
        return random.choice(REPONSES_SOCIALES["Mok"])

    

    return None

def chatbot_ensa(query):
    # ── ÉTAPE 0 : Détection messages sociaux ─────────────────
    reponse_sociale = detecter_social(query)
    if reponse_sociale:
        return reponse_sociale, 99, "social", "Built-in social response."

    # ── ÉTAPE 1 : Retrieval ───────────────────────────────────
    passages = retrieve(query)
    if not passages:
        return "Internal error.", 0, "error", ""

    meilleur = passages[0]

    # ── ÉTAPE 2 : Vérification seuil ─────────────────────────
    if meilleur["score_cosinus"] < SEUIL_COSINUS:
        return (
            "Information not available in my ENSA Fès database. "
            "Please contact contact@ensa-fes.ma",
            0, "hors_kb", "No relevant passage found."
        )

    # ── ÉTAPE 3 : Extraction RoBERTa ─────────────────────────
    contexte = meilleur["answer"]
    reponse, score_qa = read_answer(query, contexte)
    confiance = calculer_confiance(meilleur["score_cosinus"], score_qa)

    return (
        reponse,
        confiance,
        meilleur["category"],
        meilleur["answer"]
    )

print("Pipeline prêt !")

Pipeline prêt !


In [36]:
def repondre(question, historique):
    if not question.strip():
        return historique, "", "", ""

    reponse, confiance, categorie, passage = chatbot_ensa(question)

    if confiance >= 70:
        couleur = "#1D9E75"
        label = f"High confidence — {confiance}%"
        icone = "✓"
    elif confiance >= 40:
        couleur = "#e67e22"
        label = f"Moderate confidence — {confiance}%"
        icone = "~"
    else:
        couleur = "#c0392b"
        label = f"Low confidence — {confiance}%"
        icone = "!"

    badge_html = f"""
    <div style="margin-top:10px;">
        <div style="display:inline-flex;align-items:center;gap:8px;
            padding:8px 18px;background:{couleur};color:white;
            border-radius:24px;font-size:13px;font-weight:600;">
            <span>{icone}</span><span>{label}</span>
        </div>
        <div style="margin-top:6px;font-size:12px;color:#5a7fa8;
            padding-left:4px;">
            Category : <strong>{categorie}</strong>
        </div>
    </div>
    """

    historique = historique or []
    historique.append({"role": "user",      "content": question})
    historique.append({"role": "assistant", "content": reponse})
    return historique, badge_html, passage, ""


css = """
/* ── Global ───────────────────────────── */
.gradio-container {
    max-width: 880px !important;
    margin: auto !important;
    background: #f0f4f9 !important;
}

/* ── Header ───────────────────────────── */
#ensa-header {
    background: linear-gradient(135deg, #003580 0%, #0057b8 100%);
    border-radius: 16px;
    padding: 28px 32px 22px 32px;
    margin-bottom: 18px;
    color: white;
    text-align: center;
}
#ensa-header .badge {
    display: inline-block;
    background: rgba(255,255,255,0.15);
    border: 1px solid rgba(255,255,255,0.3);
    color: white;
    font-size: 11px;
    font-weight: 700;
    padding: 4px 14px;
    border-radius: 20px;
    letter-spacing: 2px;
    margin-bottom: 12px;
}
#ensa-header h1 {
    font-size: 28px;
    font-weight: 700;
    margin: 0 0 6px 0;
    color: white;
}
#ensa-header p {
    font-size: 13px;
    color: rgba(255,255,255,0.75);
    margin: 0;
}

/* ── Suggestion buttons ───────────────── */
.sugg button {
    background: white !important;
    border: 1.5px solid #ccdcf0 !important;
    border-radius: 24px !important;
    color: #003580 !important;
    font-size: 12px !important;
    font-weight: 500 !important;
    padding: 6px 14px !important;
    transition: all 0.2s !important;
    box-shadow: 0 1px 3px rgba(0,53,128,0.08) !important;
}
.sugg button:hover {
    background: #003580 !important;
    color: white !important;
    border-color: #003580 !important;
}

/* ── Chatbot ──────────────────────────── */
#chatwindow {
    border-radius: 16px !important;
    border: 1.5px solid #ccdcf0 !important;
    background: white !important;
}

/* ── Input row ────────────────────────── */
#input-row {
    background: white;
    border-radius: 14px;
    border: 1.5px solid #ccdcf0;
    padding: 6px 6px 6px 14px;
    display: flex;
    align-items: center;
    margin-top: 10px;
    box-shadow: 0 2px 8px rgba(0,53,128,0.07);
}

/* ── Send button ──────────────────────── */
#send-btn {
    background: #003580 !important;
    color: white !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    padding: 10px 22px !important;
    border: none !important;
    transition: background 0.2s !important;
}
#send-btn:hover {
    background: #0057b8 !important;
}

/* ── Clear button ─────────────────────── */
#clear-btn button {
    color: #8da9c4 !important;
    font-size: 12px !important;
    border: 1px solid #e0eaf5 !important;
    border-radius: 8px !important;
    background: white !important;
}

/* ── Accordion ────────────────────────── */
.gr-accordion {
    border-radius: 12px !important;
    border: 1.5px solid #ccdcf0 !important;
    background: white !important;
}

/* ── Footer ───────────────────────────── */
#ensa-footer {
    text-align: center;
    font-size: 12px;
    color: #8da9c4;
    margin-top: 18px;
    padding-top: 14px;
    border-top: 1.5px solid #e0eaf5;
}
"""

with gr.Blocks(css=css) as demo:

    # ── Header ───────────────────────────────────────────────
    gr.HTML("""
    <div id="ensa-header">
        <div class="badge">ENSA FÈS · AI ASSISTANT</div>
        <h1>Intelligent Chatbot</h1>
        <p>RAG Pipeline &nbsp;·&nbsp; multi-qa-MiniLM &nbsp;·&nbsp; RoBERTa
        &nbsp;·&nbsp; Ask anything about ENSA Fès</p>
    </div>
    """)

    # ── Suggestions ──────────────────────────────────────────
    gr.HTML("<p style='font-size:12px;color:#5a7fa8;margin:0 0 8px 4px;font-weight:500;'>Quick questions :</p>")
    with gr.Row(elem_classes="sugg"):
        btn_s1 = gr.Button("Admission process",    size="sm")
        btn_s2 = gr.Button("Engineering programs", size="sm")
        btn_s3 = gr.Button("Internships & PFE",    size="sm")
    with gr.Row(elem_classes="sugg"):
        btn_s4 = gr.Button("Student clubs",   size="sm")
        btn_s5 = gr.Button("Campus & contacts", size="sm")
        btn_s6 = gr.Button("Tuition fees",    size="sm")

    # ── Chatbot ──────────────────────────────────────────────
    chatbot = gr.Chatbot(
        label="",
        height=400,
        show_label=False,
        elem_id="chatwindow"
    )

    # ── Input ────────────────────────────────────────────────
    with gr.Row():
        txt_input = gr.Textbox(
            placeholder="Ask your question about ENSA Fès...",
            show_label=False,
            scale=5,
            container=False
        )
        btn_send = gr.Button(
            "Send",
            scale=1,
            elem_id="send-btn"
        )

    # ── Confidence ───────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=4):
            conf_html = gr.HTML(
                value="<div style='color:#8da9c4;font-size:13px;"
                      "padding:10px 4px;'>Confidence will appear after your first question.</div>"
            )
        with gr.Column(scale=1, elem_id="clear-btn"):
            btn_reset = gr.Button("Clear chat", size="sm")

    # ── Source ───────────────────────────────────────────────
    with gr.Accordion("Retrieved source passage", open=False):
        passage_out = gr.Textbox(
            label="",
            lines=3,
            interactive=False,
            placeholder="The retrieved context passage will appear here..."
        )

    # ── Footer ───────────────────────────────────────────────
    gr.HTML("""
    <div id="ensa-footer">
        ENSA Fès Intelligent Chatbot &nbsp;·&nbsp;
        RAG · BERT + RoBERTa &nbsp;·&nbsp;
        École Nationale des Sciences Appliquées de Fès · 2024–2025
    </div>
    """)

    # ── State ────────────────────────────────────────────────
    state = gr.State([])

    # ── Events ───────────────────────────────────────────────
    def envoyer(question, historique):
        return repondre(question, historique)

    btn_send.click(
        envoyer,
        inputs=[txt_input, state],
        outputs=[chatbot, conf_html, passage_out, txt_input]
    )
    txt_input.submit(
        envoyer,
        inputs=[txt_input, state],
        outputs=[chatbot, conf_html, passage_out, txt_input]
    )

    btn_s1.click(lambda: "How to get admitted to ENSA Fes?",        outputs=txt_input)
    btn_s2.click(lambda: "What engineering programs are available?", outputs=txt_input)
    btn_s3.click(lambda: "Are internships mandatory at ENSA Fes?",  outputs=txt_input)
    btn_s4.click(lambda: "What clubs are available at ENSA Fes?",   outputs=txt_input)
    btn_s5.click(lambda: "Where is ENSA Fes located?",              outputs=txt_input)
    btn_s6.click(lambda: "What are the tuition fees at ENSA Fes?",  outputs=txt_input)

    btn_reset.click(
        lambda: (
            [],
            [],
            "<div style='color:#8da9c4;font-size:13px;padding:10px 4px;'>"
            "Confidence will appear after your first question.</div>"
        ),
        outputs=[chatbot, state, conf_html]
    )

demo.launch(
    share=False,
    server_port=7870,
    show_error=True
)

C:\Users\BrandlyFES\AppData\Local\Temp\ipykernel_3304\1693889061.py:159: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css) as demo:


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
